# ICR Models Comparisson

## Import data and modules

In [1]:
import numpy as np # linear algebra
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss, balanced_accuracy_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv
/kaggle/input/icr-identify-age-related-conditions/greeks.csv
/kaggle/input/icr-identify-age-related-conditions/train.csv
/kaggle/input/icr-identify-age-related-conditions/test.csv


In [2]:
random_seed = 420

# Load the train.csv data
train = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv', index_col='Id')

# Load the test.csv data
test = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv', index_col='Id')


## Preprocessing

In [3]:
# Split the data into features and target
X = train.drop('Class', axis=1)
y = train['Class']

# Convert categorical feature to one-hot encoding
X = pd.get_dummies(X, columns=['EJ'], drop_first=True)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=random_seed)


In [4]:
# Define the preprocessing steps
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # Use mean strategy to fill missing values
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Use most frequent strategy to fill missing values
    ('encoder', OneHotEncoder(sparse=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

## Models

### Logistic Regression

In [5]:
# Create a pipeline with preprocessing and Random Forest Classifier
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', LogisticRegression())])


In [6]:
# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ', 'BC', 'BD ', 'BN', 'BP',
       'BQ', 'BR', 'BZ', 'CB', 'CC', 'CD ', 'CF', 'CH', 'CL', 'CR', 'CS', 'CU',
       'CW ', 'DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY', 'EB',
       'EE', 'EG', 'EH', 'EL', 'EP', 'EU', 'FC', 'FD ', 'FE', 'FI', 'FL', 'FR',
       'FS', 'GB', 'GE', 'GF', 'GH', 'GI', 'GL'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse=False))]),
                                                  Index([], dtype='object'))])),
                ('classifier', LogisticRegression())])

In [7]:
# Make predictions on the validation set
val_predictions = pipeline.predict(X_val)
val_probabilities = pipeline.predict_proba(X_val)[:, 1]  # Probability estimates for the positive class


In [8]:
# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)


Validation Log Loss: 0.33434545040004704
Validation Balanced Log Loss: 0.4883064856461277
Validation Balanced Accuracy: 0.7754260582737769


### Random Forest Classifier

In [9]:
from sklearn.linear_model import LogisticRegression

# Create a pipeline with preprocessing and logistic regression
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', RandomForestClassifier(random_state=random_seed))])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the validation set
val_predictions = pipeline.predict(X_val)
val_probabilities = pipeline.predict_proba(X_val)[:, 1]  # Probability estimates for the positive class

# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)


Validation Log Loss: 0.22287640727310543
Validation Balanced Log Loss: 0.41898856935257195
Validation Balanced Accuracy: 0.6871907641561297


### Support Vector Machines Classifier

In [10]:
from sklearn.svm import SVC

# Create a pipeline with preprocessing and SVM classifier
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', SVC(probability=True, random_state=random_seed))])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the validation set
val_probabilities = pipeline.predict_proba(X_val)[:, 1]  # Probability estimates for the positive class

# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)


Validation Log Loss: 0.24799941555252106
Validation Balanced Log Loss: 0.46657393503495226
Validation Balanced Accuracy: 0.6871907641561297


### XGBoost Classifier

In [11]:
import xgboost as xgb

# Create a pipeline with preprocessing and XGBoost classifier
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', xgb.XGBClassifier(random_state=random_seed))])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the validation set
val_probabilities = pipeline.predict_proba(X_val)[:, 1]  # Probability estimates for the positive class

# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)

Validation Log Loss: 0.1871711075048476
Validation Balanced Log Loss: 0.4440115758076805
Validation Balanced Accuracy: 0.6871907641561297


### LightGBM Classifier

In [12]:
import lightgbm as lgb

# Create a pipeline with preprocessing and LightGBM classifier
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', lgb.LGBMClassifier(random_state=random_seed))])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the validation set
val_probabilities = pipeline.predict_proba(X_val)[:, 1]  # Probability estimates for the positive class

# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)

Validation Log Loss: 0.21728882666453345
Validation Balanced Log Loss: 0.5438228711211811
Validation Balanced Accuracy: 0.6871907641561297


### CatBoost Classifier

In [13]:
import catboost as cb

# Create a pipeline with preprocessing and CatBoost classifier
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', cb.CatBoostClassifier(random_state=random_seed, verbose=False))])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the validation set
val_probabilities = pipeline.predict_proba(X_val)[:, 1]  # Probability estimates for the positive class

# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)

Validation Log Loss: 0.16562570935341436
Validation Balanced Log Loss: 0.38160157597551764
Validation Balanced Accuracy: 0.6871907641561297


### Neural Network 

In [14]:
import tensorflow as tf
from tensorflow import keras

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])



In [15]:
# Create a pipeline with preprocessing
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Preprocess the training and validation data
X_train_preprocessed = pipeline.fit_transform(X_train)
X_val_preprocessed = pipeline.transform(X_val)


# Define the neural network model
model = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(X_train_preprocessed.shape[1],)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy')

# Fit the model on the training data
model.fit(X_train_preprocessed, y_train, epochs=10, batch_size=32, verbose=0)

# Make predictions on the validation set
val_probabilities = model.predict(X_val_preprocessed).flatten()  # Probability estimates for the positive class

# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)

4/4 [==============================] - 0s 3ms/step
Validation Log Loss: 0.26327384382461716
Validation Balanced Log Loss: 0.519516847419502
Validation Balanced Accuracy: 0.6871907641561297


### Naive Bayes Classifier

In [16]:
from sklearn.naive_bayes import GaussianNB

# Create a pipeline with preprocessing and Naive Bayes classifier
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', GaussianNB())])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the validation set
val_probabilities = pipeline.predict_proba(X_val)[:, 1]  # Probability estimates for the positive class

# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)

Validation Log Loss: 3.9392424832089303
Validation Balanced Log Loss: 8.44969273997643
Validation Balanced Accuracy: 0.6871907641561297


### K-Nearest Neighbour Classifier

In [17]:
from sklearn.neighbors import KNeighborsClassifier

# Create a pipeline with preprocessing and KNN classifier
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', KNeighborsClassifier(n_neighbors=5))])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Make predictions on the validation set
val_probabilities = pipeline.predict_proba(X_val)[:, 1]  # Probability estimates for the positive class

# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)

Validation Log Loss: 1.901721743486446
Validation Balanced Log Loss: 5.229795333198141
Validation Balanced Accuracy: 0.6871907641561297


## Optuna Hyper-parameter Search

In [18]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, make_scorer
from sklearn.model_selection import cross_val_score

# Define the objective function for Optuna
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'seed': random_seed,
        'tree_method': 'gpu_hist',  # Use GPU for faster training (if available)
        'verbosity': 0
    }
    
    # Define the hyperparameters to search and their ranges
    param_grid = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.001, 0.1),
        'subsample': trial.suggest_uniform('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-6, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-6, 10.0),
    }

    model = xgb.XGBClassifier(**params, **param_grid)
    
    # Train and evaluate the model using cross-validation
    scores = -cross_val_score(model, X_train, y_train, cv=3, scoring=make_scorer(log_loss))
    return scores.mean()

# Create a study object and optimize the objective function
study_name = 'xgb_search'
study = optuna.create_study(
    study_name=study_name,
    storage="sqlite:///"+study_name+".db",
    direction='minimize',
    load_if_exists=True
)
study.optimize(objective, n_trials=100)

[I 2023-06-08 02:02:58,313] A new study created in RDB with name: xgb_search
[I 2023-06-08 02:03:02,791] Trial 0 finished with value: -3.365939102523985 and parameters: {'n_estimators': 600, 'max_depth': 9, 'learning_rate': 0.0015636261611639774, 'subsample': 0.6692100779620073, 'colsample_bytree': 0.6874408967869192, 'reg_alpha': 0.19151789600036326, 'reg_lambda': 5.4064061166302766e-06}. Best is trial 0 with value: -3.365939102523985.
[I 2023-06-08 02:03:07,273] Trial 1 finished with value: -3.5848294834162586 and parameters: {'n_estimators': 900, 'max_depth': 5, 'learning_rate': 0.0010893622609624946, 'subsample': 0.9609729654475938, 'colsample_bytree': 0.7232996450979168, 'reg_alpha': 2.6599932734516176e-06, 'reg_lambda': 0.0021877996044996403}. Best is trial 1 with value: -3.5848294834162586.
[I 2023-06-08 02:03:08,419] Trial 2 finished with value: -3.0715692799447214 and parameters: {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.018959550498179132, 'subsample': 0.605301

In [19]:
# Get the best hyperparameters and the best score
best_params = study.best_params
best_score = study.best_value

print("Best Hyperparameters:", best_params)
print("Best Score:", best_score)

Best Hyperparameters: {'colsample_bytree': 0.6511799087171619, 'learning_rate': 0.002272349895884903, 'max_depth': 3, 'n_estimators': 100, 'reg_alpha': 9.832657785639702, 'reg_lambda': 7.572104555587343, 'subsample': 0.8738349691592425}
Best Score: -6.140918576310413


In [20]:
# Train the final model with the best hyperparameters
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6511799087171619, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.002272349895884903,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=100, n_jobs=None, num_parallel_tree=None,
              predictor=None, random_state=None, ...)

In [21]:
# Make predictions on the validation set
val_probabilities = final_model.predict_proba(X_val)[:, 1]  # Probability estimates for the positive class

# Calculate log loss
logloss = log_loss(y_val, val_probabilities)
print("Validation Log Loss:", logloss)

# Calculate balanced log loss (using predicted probabilities)
balanced_logloss = log_loss(y_val, val_probabilities, sample_weight=y_val.map({0: 1, 1: 5}))  # Adjust sample weights to balance classes
print("Validation Balanced Log Loss:", balanced_logloss)

# Calculate balanced accuracy
balanced_accuracy = balanced_accuracy_score(y_val, val_predictions)
print("Validation Balanced Accuracy:", balanced_accuracy)

Validation Log Loss: 0.5923982247621447
Validation Balanced Log Loss: 0.6496387286735584
Validation Balanced Accuracy: 0.6871907641561297


## Submission

In [22]:
# Preprocess the test data
# Convert categorical feature to one-hot encoding
test = pd.get_dummies(test, columns=['EJ'], drop_first=True)

# Ensure that the test data has the same columns as the training data
missing_cols = set(X.columns) - set(test.columns)
for col in missing_cols:
    test[col] = 0
test = test[X.columns]

# Make predictions on the test data
test_probabilities = final_model.predict_proba(test)[:, 1]  # Probability estimates for the positive class


In [23]:
# Create submission DataFrame
submission = pd.DataFrame({'Id': test.index, 'class_0': 1 - test_probabilities, 'class_1': test_probabilities})

# Save the submission DataFrame to a CSV file
submission.to_csv('submission.csv', index=False)